In [ ]:
# Assignment Notebook (alias-based, backend-agnostic test)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()
print("Backend:", m.backend_name)

# -------------------------------------------------------------------------------
# Problem data
# -------------------------------------------------------------------------------
rng = np.random.default_rng(42)

# Number of workers and jobs
n = 8

# Random assignment costs
cost = rng.integers(10, 100, size=(n, n))

print("Assignment cost matrix:")
print(cost)

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
x = [
    [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n)]
    for i in range(n)
]

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

# Each worker performs exactly one job
for i in range(n):
    m.add_constraint(
        quicksum(x[i][j] for j in range(n)) == 1
    )

# Each job is assigned to exactly one worker
for j in range(n):
    m.add_constraint(
        quicksum(x[i][j] for i in range(n)) == 1
    )

# -------------------------------------------------------------------------------
# Objective
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(
        cost[i, j] * x[i][j]
        for i in range(n)
        for j in range(n)
    ),
    GRB.MINIMIZE,
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
assignment = np.array([[v.x for v in row] for row in x])

print("\nAssignment matrix:")
print(assignment.astype(int))

print("\nAssignments:")
for i in range(n):
    j = np.argmax(assignment[i])
    print(f"Worker {i:2d} -> Job {j:2d}   Cost = {cost[i, j]}")

print("\nTotal cost:", np.sum(cost * assignment))